In [1]:
pip install pandas reportlab

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 692.7 kB/s eta 0:00:03
   ---------- ----------------------------- 0.5/2.0 MB 692.7 kB/s eta 0:00:03
   ---------------- ----------------------- 0.8/2.0 MB 571.3 kB/s eta 0:00:03
   ---------------- ----------------------- 0.8/2.0 MB 571.3 kB/s eta 0:00:03
   ---------------- ----------------------- 0.8/2.0 MB 571.3 kB/s eta 0:00:03
   --------------------- ------------------ 1.0/2.0 MB 512.1 kB/s eta 0:00:02
   --------------------- ------------------ 1.0/2.0 MB 512.1 kB/s eta 0:00:02
   --------------------- ---------------

In [3]:
import pandas as pd 
def generate_phase_profile(csv_path,target_player): 
    df = pd.read_csv(csv_path) 
    results={}
    
    phases = ['Powerplay(0-5)','Middle(6-14)','Death(15-19)'] 
    
    def get_phase(over):
      if over < 6:
        return 'Powerplay(0-5)' 
      elif over < 15: 
        return 'Middle(6-14)' 
      else: 
        return 'Death(15-19)' 
          
    batter_df = df[(df['striker'] == target_player) & (df['wides'].isna())].copy() 

    if not batter_df.empty: 
        batter_df['over_num'] = batter_df['ball'].astype(int) 
        batter_df['phase'] = batter_df['over_num'].apply(get_phase)

        bat_summary = [] 

        for phase in phases: 
            p_df = batter_df[batter_df['phase'] == phase]
            balls_faced = len(p_df) 
            runs_scored = p_df['runs_off_bat'].sum() 
            dot_balls = len(p_df[p_df['runs_off_bat'] == 0]) 
            boundaries = len(p_df[p_df['runs_off_bat'].isin([4,6])]) 

            strike_rate = round((runs_scored / balls_faced * 100), 2) if balls_faced > 0 else 0.0 
            dot_percentage = round((dot_balls / balls_faced * 100), 2) if balls_faced > 0 else 0.0 
            boundary_percentage = round((boundaries / balls_faced * 100), 2) if balls_faced > 0 else 0.0 

            wickets_lost = len(p_df[p_df['player_dismissed'] == target_player])
            
            bat_summary.append({
               'Phase' : phase,
               'Runs' : runs_scored,
               'Balls' : balls_faced,
               'Dismissals' : wickets_lost,
               'Strike Rate' : strike_rate, 
               'Dot %' : dot_percentage,
               'Boundary %' : boundary_percentage
           }) 
        results['Batting'] = pd.DataFrame(bat_summary)

    bowler_df = df[(df['bowler'] == target_player) & (df['wides'].isna()) & (df['noballs'].isna())].copy() 
    if not bowler_df.empty : 
        bowler_df['over_num'] = bowler_df['ball'].astype(int) 
        bowler_df['phase'] = bowler_df['over_num'].apply(get_phase)

        bowler_summary = [] 

        for phase in phases : 
            p_df = bowler_df[bowler_df['phase'] == phase]
            balls_faced = len(p_df) 
            runs_conceded = p_df['runs_off_bat'].sum() + p_df['extras'].sum() 
            dot_balls = len(p_df[p_df['runs_off_bat'] == 0]) 

            wickets_taken = len(p_df[p_df['wicket_type'].notna() & (p_df['wicket_type'].isin(['run out']))])   
            econ = round((runs_conceded / (balls_faced / 6)), 2) if balls_faced > 0 else 0.0
            dot_percentage = round((dot_balls / balls_faced * 100), 2) if balls_faced > 0 else 0.0 
            bowler_summary.append({
                  'Phase' : phase,
                  'Runs Conceded' : int(runs_conceded),
                  'Balls' : balls_faced,
                  
                  'Wickets' : wickets_taken,
                  'Economy' : econ,
                  'Dot %' : dot_percentage,
            }) 
        results['Bowling'] = pd.DataFrame(bowler_summary)

        return results if results else f"No records found for {target_player} in this match." 
profile_data = generate_phase_profile("extracted_ipl_data/1535463.csv","PJ Cummins") 

if isinstance(profile_data , dict): 
    for role , table in profile_data.items(): 
        print(f"\n--- {role} Profile ---")
        display(table)
else:
    print(profile_data)
    


--- Batting Profile ---


,Phase,Runs,Balls,Dismissals,Strike Rate,Dot %,Boundary %
0,Powerplay(0-5),0,0,0,0.0,0.0,0.0
1,Middle(6-14),1,2,1,50.0,50.0,0.0
2,Death(15-19),0,0,0,0.0,0.0,0.0



--- Bowling Profile ---


,Phase,Runs Conceded,Balls,Wickets,Economy,Dot %
0,Powerplay(0-5),33,12,0,16.5,16.67
1,Middle(6-14),17,6,0,17.0,16.67
2,Death(15-19),13,6,0,13.0,0.00


In [4]:
#goal= quantify how a batter responds to defensive bowling pressure (consecutive dot balls vs bounday free release opportunity)
#code= scans delievery seq to track dot ball streaks and measure how often a batter breaks under pressure with a boundary on the subsequent delivery


import pandas as pd 
def calculate_pressure_metrics(csv_path,target_batter): 
    df = pd.read_csv(csv_path) 

    b_df = df[(df['striker'] == target_batter) & (df['wides'].isna())].copy() 
    b_df = b_df.sort_values(by = ['innings','ball']) 

    b_df['is_dot'] = (b_df['runs_off_bat'] == 0)
    b_df['is_boundary'] = b_df['runs_off_bat'].isin([4,6]) 

    total_balls = len(b_df) 
    total_dots = b_df['is_dot'].sum() 
    total_boundaries = b_df['is_boundary'].sum() 

    b_df['prev_was_dot'] = b_df['is_dot'].shift(1).fillna(False) 
    release_shots = len(b_df[b_df['prev_was_dot'] & b_df['is_boundary']])

    metrics={
        "Total Legal Balls" : total_balls, 
        "Total Dot Balls" : total_dots, 
        "Dot Ball %" : round((total_dots / total_balls * 100), 2) if total_balls > 0 else 0, 
        "Pressure Release Shots" : release_shots, 
        "Release Efficiency %" : round((release_shots / total_dots * 100), 2) if total_dots >0 else 0 
    }
    return pd.DataFrame([metrics]) 

pressure_df = calculate_pressure_metrics("extracted_ipl_data/1535463.csv","Dhruv Jurel") 
display(pressure_df)

,Total Legal Balls,Total Dot Balls,Dot Ball %,Pressure Release Shots,Release Efficiency %
0,21,3,14.29,1,33.33


In [1]:
df

NameError: name 'df' is not defined

In [5]:
df.head()

NameError: name 'df' is not defined